# 4: Dask — Scalable Raster Analytics

Open a COG with chunking, compute NDVI lazily, and run a spatial reduction that only touches the needed chunks.

**Dependencies:** `rioxarray`, `xarray`, `pystac-client`, `planetary-computer`

In [ ]:
import rioxarray
import xarray as xr
import pystac_client
import planetary_computer

## Get signed URLs for red and NIR

In [ ]:
# Northern Karnataka, India (same study area as Notebook 1)
BBOX = [75.62, 16.20, 77.29, 17.47]  # [min_lon, min_lat, max_lon, max_lat]

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1"
)
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=BBOX,
    datetime="2023-06-01/2023-06-30",
    max_items=1,
)
item = planetary_computer.sign(list(search.items())[0])
url_red = item.assets["B04"].href
url_nir = item.assets["B08"].href

## Open both bands with chunks (Dask-backed)

Same chunk layout so that arithmetic aligns.

In [ ]:
chunks = {"x": 1024, "y": 1024}
red = rioxarray.open_rasterio(url_red, chunks=chunks).squeeze("band", drop=True)
nir = rioxarray.open_rasterio(url_nir, chunks=chunks).squeeze("band", drop=True)

ds = xr.Dataset({"red": red, "nir": nir})
print("Chunks:", ds["red"].chunks)

## Compute NDVI (lazy)

No data has been read yet; the expression builds a Dask graph.

In [ ]:
ds["ndvi"] = (ds["nir"] - ds["red"]) / (ds["nir"] + ds["red"])
ndvi = ds["ndvi"]
print("NDVI is lazy:", ndvi.data)

## Spatial mean (reduction)

`.mean(dim=["x", "y"])` reduces to a scalar. Only then we call `.compute()` to run the graph.

In [ ]:
mean_ndvi = ndvi.mean(dim=["x", "y"])
result = mean_ndvi.compute()
print("Mean NDVI (computed):", float(result))

## Optional: clip then reduce

Clip to a smaller box so less data is read; then compute mean.

In [ ]:
ds_clip = ds.rio.clip_box(
    minx=76.40, miny=16.75, maxx=76.55, maxy=16.90, crs="EPSG:4326"
)
mean_clip = ds_clip["ndvi"].mean(dim=["x", "y"]).compute()
print("Mean NDVI (clipped region):", float(mean_clip))